# Dallas 311 Intelligence — GPU Training ALL Models (Colab)
**500,000 rows · RAPIDS cuML & XGBoost CUDA**

## Instructions
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Mount Google Drive and place your CSV at `MyDrive/Dallas311/dallas_311_500k.csv`
3. Run all cells in order
4. Download the artifacts from `MyDrive/Dallas311/artifacts/`

In [ ]:
# ── Cell 1: Verify GPU ──────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU detected — change runtime type!')

In [ ]:
# ── Cell 2: Install dependencies (RAPIDS cuML + XGBoost) ────────────────────
print('Installing RAPIDS libraries for hardware-accelerated training...')
!pip install -q --extra-index-url=https://pypi.nvidia.com cudf-cu12 cuml-cu12
!pip install -q xgboost scikit-learn pandas numpy joblib
print('Ready.')

In [ ]:
# ── Cell 3: Mount Google Drive ───────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT  = '/content/drive/MyDrive/Dallas311'
DATA_PATH   = f'{DRIVE_ROOT}/dallas_311_500k.csv'
ARTIFACT_DIR = f'{DRIVE_ROOT}/artifacts'
os.makedirs(ARTIFACT_DIR, exist_ok=True)
print(f'Data path : {DATA_PATH}')
print(f'Artifact  : {ARTIFACT_DIR}')

In [ ]:
# ── Cell 4: Load data ────────────────────────────────────────────────────────
import pandas as pd
import numpy as np

DTYPE_MAP = {
    'Department': 'category',
    'Priority': 'category',
    'Method Received Description': 'category',
    'City Council District': 'category',
    'Service Request Type': 'category'
}

print('Loading CSV...')
df = pd.read_csv(DATA_PATH, dtype=DTYPE_MAP, low_memory=False)
print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols')
print(df.dtypes)

In [ ]:
# ── Cell 5: Cleaning & Feature Engineering ───────────────────────────────────

# Drop non-informative columns
DROP_INITIAL = ['Service Request Number', 'Unique Key', 'Address']
df.drop(columns=[c for c in DROP_INITIAL if c in df.columns], inplace=True)

# Group rare departments
MIN_DEPT_COUNT = 1000
dept_counts = df['Department'].value_counts()
rare_depts  = dept_counts[dept_counts < MIN_DEPT_COUNT].index
df['Department_grouped'] = df['Department'].cat.add_categories('Other')
df.loc[df['Department'].isin(rare_depts), 'Department_grouped'] = 'Other'

# Parse dates
DATE_FORMAT = '%Y %b %d %I:%M:%S %p'
for col in ['Created Date', 'Closed Date']:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], format=DATE_FORMAT, errors='coerce')

# Response time + binary target
df['days_to_close'] = (df['Closed Date'] - df['Created Date']).dt.total_seconds() / 3600
df = df[df['days_to_close'] > 0].copy()
df['target'] = (df['days_to_close'] <= 72).astype(int)

# Time features
df['hour']        = df['Created Date'].dt.hour
df['day_of_week'] = df['Created Date'].dt.dayofweek
df['month']       = df['Created Date'].dt.month

print(f'After cleaning: {df.shape[0]:,} rows')
print(df['target'].value_counts(normalize=True).round(3))

In [ ]:
# ── Cell 6: Preprocessing ────────────────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder, StandardScaler
import joblib

# Drop leakage + processed columns
LEAKAGE = ['Closed Date', 'Update Date', 'Status', 'Outcome',
           'Lat_Long Location', 'Overall Service Request Due Date',
           'Created Date', 'days_to_close', 'Department']
df.drop(columns=[c for c in LEAKAGE if c in df.columns], inplace=True)

# ERT: extract numeric days
if 'Estimated Response Time Description' in df.columns:
    df['ERT_days'] = df['Estimated Response Time Description'].str.extract(r'(\d+)').astype(float)
    df.drop(columns=['Estimated Response Time Description'], inplace=True)

# Cast categoricals to string to allow missing categories
for col in df.select_dtypes(include='category').columns:
    df[col] = df[col].astype(str)

# Service Request Type — keep top 15
TOP_N = 15
if 'Service Request Type' in df.columns:
    top_types = df['Service Request Type'].value_counts().nlargest(TOP_N).index
    df['Service Request Type'] = df['Service Request Type'].where(
        df['Service Request Type'].isin(top_types), 'Other'
    )

# Label Encoding
ENCODE_COLS = ['Priority', 'Method Received Description', 'Department_grouped']
encoders = {}
for col in ENCODE_COLS:
    if col in df.columns:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        encoders[col] = le

# One-hot encode remaining categoricals
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
cat_cols = [c for c in cat_cols if c != 'target']
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# Split
from sklearn.model_selection import train_test_split
X = df.drop(columns=['target'])
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Handle missing values
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='median')
X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
X_test  = pd.DataFrame(imputer.transform(X_test),      columns=X_test.columns)

# Scale
scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test  = pd.DataFrame(scaler.transform(X_test),      columns=X_test.columns)

# Save feature list (critical for inference alignment)
feature_names = list(X_train.columns)
joblib.dump(feature_names, f'{ARTIFACT_DIR}/feature_names.pkl')
joblib.dump(imputer,       f'{ARTIFACT_DIR}/imputer.pkl')
joblib.dump(scaler,        f'{ARTIFACT_DIR}/scaler.pkl')
joblib.dump(encoders,      f'{ARTIFACT_DIR}/label_encoders.pkl')
print(f'Feature count: {len(feature_names)}')
print('Preprocessors saved.')

In [ ]:
# ── Cell 7: Train Logistic Regression on GPU (cuML) ──────────────────────────
from cuml.linear_model import LogisticRegression
import time
import joblib

log_model = LogisticRegression(max_iter=2000)

print('Training Logistic Regression on GPU via RAPIDS...')
t0 = time.time()
log_model.fit(X_train.astype('float32'), y_train.astype('float32'))
print(f'Done in {time.time()-t0:.2f}s')

# Save
joblib.dump(log_model, f'{ARTIFACT_DIR}/logistic_regression_500k.joblib')

In [ ]:
# ── Cell 8: Train Random Forest on GPU (cuML) ───────────────────────────────
from cuml.ensemble import RandomForestClassifier
import time
import joblib

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42
)

print('Training Random Forest on GPU via RAPIDS...')
t0 = time.time()
rf_model.fit(X_train.astype('float32'), y_train.astype('int32'))
print(f'Done in {time.time()-t0:.2f}s')

# Save
joblib.dump(rf_model, f'{ARTIFACT_DIR}/random_forest_500k.joblib')

In [ ]:
# ── Cell 9: Train XGBoost on CUDA GPU ───────────────────────────────────────
from xgboost import XGBClassifier
import time

xgb_model = XGBClassifier(
    n_estimators=1000,        # Room for more trees with slower LR
    max_depth=6,
    learning_rate=0.03,
    reg_alpha=0.1,            # L1 regularization
    reg_lambda=2.0,           # L2 regularization
    min_child_weight=10,      # min samples per leaf
    gamma=0.1,                # split sensitivity
    subsample=0.8,
    colsample_bytree=0.8,
    early_stopping_rounds=50, # patient stopping criteria
    eval_metric='logloss',
    device='cuda',
    tree_method='hist',
    random_state=42,
)

print('Training XGBoost on GPU with regularization...')
t0 = time.time()
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=100
)
elapsed = time.time() - t0
print(f'\nXGBoost complete in {elapsed:.1f}s')

# Save
xgb_model.save_model(f'{ARTIFACT_DIR}/xgb_model_500k.json')

In [ ]:
# ── Cell 10: Evaluate and Compare ───────────────────────────────────────────
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
import json

models = {
    'Logistic Regression': log_model,
    'Random Forest': rf_model,
    'XGBoost': xgb_model
}

results_bundle = {}

for name, model in models.items():
    print(f'=== Evaluating {name} ===')
    # cuML models need float32 conversion for fast validation inference
    if 'XGBoost' not in name:
        test_data = X_test.astype('float32')
    else:
        test_data = X_test
        
    y_pred      = model.predict(test_data)
    y_pred_prob = model.predict_proba(test_data)[:, 1]

    # XGBoost returns numpy, cuML can return Series
    if hasattr(y_pred, 'get'):
        y_pred = y_pred.get() # Convert to CPU numpy
    if hasattr(y_pred_prob, 'get'):
        y_pred_prob = y_pred_prob.get()

    auc  = float(roc_auc_score(y_test, y_pred_prob))
    acc  = float(accuracy_score(y_test, y_pred))
    f1   = float(f1_score(y_test, y_pred))

    print(f'ROC-AUC  : {auc:.4f}')
    print(f'Accuracy : {acc:.4f}')
    print(f'F1-Score : {f1:.4f}\n')
    
    results_bundle[name] = {
        'roc_auc': round(auc, 4),
        'accuracy': round(acc, 4),
        'f1': round(f1, 4)
    }

results_bundle['metadata'] = {
    'n_rows': len(df),
    'n_features': len(feature_names),
    'feature_names': feature_names
}

with open(f'{ARTIFACT_DIR}/results_all_gpu.json', 'w') as f:
    json.dump(results_bundle, f, indent=2)

print('All evaluations compiled and saved.')